# Training a Speech LLM with NeMo Automodel and MoE

This tutorial walks through the full **SALMAutomodel** pipeline: data preparation,
training, checkpoint conversion, and evaluation.

## What is SALMAutomodel?

SALMAutomodel is a Speech-Augmented Language Model that uses
[NeMo Automodel](https://github.com/NVIDIA-NeMo/Automodel) as the LLM backend.
It connects a pretrained ASR encoder (e.g., Canary) to a pretrained LLM via a
learnable linear projection layer.

## Why MoE?

We use **NVIDIA Nemotron Nano V3** as the LLM backbone — a Mixture-of-Experts
model with 30B total parameters but only 3B active per token. NeMo Automodel
provides two key MoE optimizations:

- **Grouped GEMM**: Fuses expert computations into a single batched matrix multiply.
- **DeepEP**: Efficient all-to-all expert routing across GPUs.

## EP and FSDP2: Same Axis

A key point: **Expert Parallelism (EP) reuses the FSDP2 data-parallel axis
(`dp_size`)**. Dense layers are sharded via FSDP2, while MoE expert layers
use EP for all-to-all routing — both on the same set of GPUs. Setting
`ep_size` does *not* add a separate dimension.

## What We Cover

1. Download Mini LibriSpeech with Lhotse
2. Train SALMAutomodel (only `perception.proj` is trained; LLM and ASR encoder are frozen)
3. Convert the distributed checkpoint to HuggingFace format
4. Evaluate with distributed inference and compute WER

## Prerequisites

- 2 GPUs
- `pip install nemo-toolkit[speechlm2]`

In [ ]:
!pip install nemo-toolkit[speechlm2]

## 1. Download Mini LibriSpeech

We use [Lhotse](https://github.com/lhotse-speech/lhotse) to download and prepare
Mini LibriSpeech — a small subset of LibriSpeech with two splits:
- `train-clean-5` (~5 hours)
- `dev-clean-2` (~2 hours)

In [1]:
from pathlib import Path
from lhotse.recipes import download_librispeech, prepare_librispeech
from lhotse import CutSet

DATA_ROOT = Path("data")
CORPUS_DIR = download_librispeech(DATA_ROOT, dataset_parts="mini_librispeech")
manifests = prepare_librispeech(
    CORPUS_DIR, dataset_parts="mini_librispeech", output_dir=DATA_ROOT / "manifests"
)

/home/pzelasko/miniconda3/envs/nemo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
data/dev-clean-2.tar.gz:   0%|                                                                              | 0.00/126M [00:00<?, ?B/s]
data/dev-clean-2.tar.gz:   0%|                                                                     | 14.3k/126M [00:00<23:05, 91.0kB/s]
data/dev-clean-2.tar.gz:   0%|                                                                      | 44.0k/126M [00:00<14:22, 146kB/s]
data/dev-clean-2.tar.gz:   0%|                                                                       | 101k/126M [00:00<08:37, 243kB/s]
data/dev-clean-2.tar.gz:   0%|                                                                       | 168k/126M [00:00<06:41, 314kB/s]
data/dev-clean-2.tar.gz:   0%|                    

## 2. Create Lhotse CutSets

Convert the raw manifests into CutSets and save as `.jsonl` files.

In [2]:
for split, parts in manifests.items():
    cuts = CutSet.from_manifests(
        recordings=parts["recordings"], supervisions=parts["supervisions"]
    )
    path = DATA_ROOT / f"cuts_{split}.jsonl"
    cuts.to_file(path)
    print(
        f"{split}: {len(cuts)} cuts, "
        f"total duration: {sum(c.duration for c in cuts):.1f}s"
    )

train-clean-5: 1519 cuts, total duration: 19118.7s
dev-clean-2: 1089 cuts, total duration: 7329.0s


## 3. Inspect the Data

Each `Cut` holds a pointer to an audio recording, its duration, and one or more
supervisions (transcripts with timing information).

In [5]:
cuts = CutSet.from_file(DATA_ROOT / "cuts_train-clean-5.jsonl")
cut = cuts[0]
print(f"ID: {cut.id}")
print(f"Duration: {cut.duration:.2f}s")
print(f"Text: {cut.supervisions[0].text}")
cut.play_audio()

ID: 669-129074-0000-0
Duration: 15.27s
Text: WHICH CONTAINS BIRTHS MARRIAGES AND DEATHS WHATEVER BECKY'S PRIVATE PLAN MIGHT BE BY WHICH DOBBIN'S TRUE LOVE WAS TO BE CROWNED WITH SUCCESS


## 4. Write Training Config

We write a YAML config for SALMAutomodel with:

- **Nemotron Nano V3** as the LLM backbone
- **EP=2** on 2 GPUs (dense layers → FSDP2, MoE layers → EP, same axis)
- Only `perception.proj` (Linear 1024→4096) is trainable; LLM, ASR encoder,
  preprocessor, and modality adapter are frozen

In [6]:
train_cuts = str((DATA_ROOT / "cuts_train-clean-5.jsonl").resolve())
val_cuts = str((DATA_ROOT / "cuts_dev-clean-2.jsonl").resolve())

config_yaml = f"""\
model:
  pretrained_llm: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  pretrained_asr: nvidia/canary-1b-flash
  pretrained_weights: true
  use_nemo_automodel: true

  prompt_format: nemotron
  audio_locator_tag: "<|audioplaceholder|>"

  freeze_params:
    - "^llm\\\\..+$"
    - "^perception\\\\.preprocessor\\\\..+$"
    - "^perception\\\\.encoder\\\\..+$"
    - "^perception\\\\.modality_adapter\\\\..+$"
  prevent_freeze_params: []

  # Uncomment to enable LoRA on the LLM:
  # lora:
  #   dim: 128
  #   alpha: 256
  #   dropout: 0.01
  #   target_modules: ["q_proj", "v_proj"]

  perception:
    target: nemo.collections.speechlm2.modules.perception.AudioPerceptionModule
    output_dim: 4096
    modality_adapter:
      _target_: nemo.collections.speechlm2.modules.perception.IdentityConnector
      d_model: 1024

  optimizer:
    _target_: torch.optim.AdamW
    lr: 5e-4
    betas: [0.9, 0.98]
    weight_decay: 1e-3
    foreach: true

  lr_scheduler:
    _target_: nemo.core.optim.lr_scheduler.CosineAnnealing
    warmup_steps: 50
    min_lr: 1e-6
    max_steps: ${{trainer.max_steps}}

trainer:
  devices: 2
  accelerator: gpu
  num_nodes: 1
  precision: bf16-true
  logger: false
  enable_checkpointing: false
  use_distributed_sampler: false
  max_steps: 100
  limit_train_batches: 50
  val_check_interval: 50
  limit_val_batches: 5
  log_every_n_steps: 10
  num_sanity_val_steps: 0
  gradient_clip_val: 1.0
  accumulate_grad_batches: 1
  strategy:
    _target_: nemo.collections.speechlm2.parts.parallel.AutomodelParallelStrategy
    dp_size: null
    dp_replicate_size: 1
    tp_size: 1
    pp_size: 1
    cp_size: 1
    ep_size: 2

data:
  train_ds:
    sample_rate: 16000
    prompt_format: ${{model.prompt_format}}
    token_equivalent_duration: 0.08
    input_cfg:
      - type: lhotse_as_conversation
        cuts_path: {train_cuts}
        audio_locator_tag: ${{model.audio_locator_tag}}
        tags:
          context: ""
    seed: 42
    shuffle: true
    shard_seed: randomized
    num_workers: 1
    batch_size: 4

  validation_ds:
    prompt_format: ${{model.prompt_format}}
    token_equivalent_duration: 0.08
    datasets:
      dev_clean_2:
        input_cfg:
          - type: lhotse_as_conversation
            cuts_path: {val_cuts}
            audio_locator_tag: ${{model.audio_locator_tag}}
            tags:
              context: "Transcribe the following:"
    sample_rate: 16000
    batch_size: 1
    seed: 42
    shard_seed: randomized

exp_manager:
  exp_dir: null
  explicit_log_dir: salm_tutorial_results/
  name: salm
  create_tensorboard_logger: false
  create_checkpoint_callback: true
  use_datetime_version: true
  resume_if_exists: true
  resume_ignore_no_checkpoint: true
  create_wandb_logger: false
  checkpoint_callback_params:
    filename: "{{step}}"
    monitor: val_loss
    mode: min
    every_n_epochs: 1
    save_top_k: 1
"""

config_path = DATA_ROOT / "salm_automodel_tutorial.yaml"
config_path.write_text(config_yaml)
print(f"Config written to {config_path.resolve()}")

Config written to /home/pzelasko/code/NeMo/worktree/speechlm2-with-nemo-automodel-merge/tutorials/speechlm2/data/salm_automodel_tutorial.yaml


## 5. Config Explained

Key settings in the config above:

| Setting | Meaning |
|---|---|
| `use_nemo_automodel: true` | Selects `SALMAutomodel` (NeMo Automodel backend) |
| `ep_size: 2` | Expert Parallelism on the FSDP data-parallel axis. Dense layers are sharded via FSDP2, MoE expert layers use EP — both on the same 2 GPUs |
| `perception.output_dim: 4096` | Nemotron Nano V3 has `hidden_size=4096` |
| `freeze_params` | Freezes the LLM, ASR encoder, preprocessor, and modality adapter. Only `perception.proj` (Linear 1024→4096) is trainable |
| `lora` (commented) | Can be enabled for parameter-efficient LLM fine-tuning using Automodel-native LoRA |

## 6. Launch Training

We use `torchrun` with 2 GPUs. The training script (`salm_train.py`) reads
`use_nemo_automodel: true` and instantiates `SALMAutomodel`.

In [10]:
import subprocess

cmd = [
    "torchrun", "--nproc_per_node=2",
    "../../examples/speechlm2/salm_train.py",
    f"--config-path={str(DATA_ROOT.resolve())}",
    "--config-name=salm_automodel_tutorial",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: torchrun --nproc_per_node=2 ../../examples/speechlm2/salm_train.py --config-path=/home/pzelasko/code/NeMo/worktree/speechlm2-with-nemo-automodel-merge/tutorials/speechlm2/data --config-name=salm_automodel_tutorial


W0317 07:22:35.934000 40985 site-packages/torch/distributed/run.py:852] 
W0317 07:22:35.934000 40985 site-packages/torch/distributed/run.py:852] *****************************************
W0317 07:22:35.934000 40985 site-packages/torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0317 07:22:35.934000 40985 site-packages/torch/distributed/run.py:852] *****************************************
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=1) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
No exporters were provided. This means that no telemetry data will be collected.
Failed to load /home/pzelasko/miniconda3/envs/nemo/lib/python3.

[NeMo I 2026-03-17 07:22:41 ear_tts_model:111] Triton available & CUDA detected. Using Triton kernel for batch_matmul.


Error executing job with overrides: []
Error executing job with overrides: []
Error locating target 'nemo.collections.speechlm2.parts.parallel.AutomodelParallelStrategy', set env var HYDRA_FULL_ERROR=1 to see chained exception.
Error locating target 'nemo.collections.speechlm2.parts.parallel.AutomodelParallelStrategy', set env var HYDRA_FULL_ERROR=1 to see chained exception.

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
[rank0]:[W317 07:22:42.369089197 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
W0317 07:22:43.739000 40985 site-packages/torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 41043 closing signal SIGTERM
E0317 07:22:43.740000 40985 site-packages/torch/distribu

CalledProcessError: Command '['torchrun', '--nproc_per_node=2', '../../examples/speechlm2/salm_train.py', '--config-path=/home/pzelasko/code/NeMo/worktree/speechlm2-with-nemo-automodel-merge/tutorials/speechlm2/data', '--config-name=salm_automodel_tutorial']' returned non-zero exit status 1.

## 7. Locate Checkpoint

In [ ]:
import glob

ckpt_paths = sorted(glob.glob("salm_tutorial_results/salm/*/checkpoints/*"))
print("Checkpoints found:", ckpt_paths)
CKPT_PATH = ckpt_paths[-1]
CONFIG_PATH = str(Path(CKPT_PATH).parent.parent / "exp_config.yaml")
print(f"Using checkpoint: {CKPT_PATH}")
print(f"Using config: {CONFIG_PATH}")

## 8. Convert Checkpoint to HuggingFace Format

Training with EP=2 produces **distributed checkpoints** (a directory with
per-rank shards). The `to_hf.py` script consolidates DTensors into regular
tensors and saves them as `config.json` + `model.safetensors`.

We launch with `torchrun --nproc_per_node=2` to match the training topology.

In [ ]:
cmd = [
    "torchrun", "--nproc_per_node=2",
    "examples/speechlm2/to_hf.py",
    "class_path=nemo.collections.speechlm2.models.SALMAutomodel",
    f"ckpt_path={CKPT_PATH}",
    f"ckpt_config={CONFIG_PATH}",
    "output_dir=salm_tutorial_hf/",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 9. Evaluate with WER

Run distributed inference on the dev set and compute Word Error Rate.
The `salm_eval.py` script prints per-batch and overall WER.

In [ ]:
DEV_CUTS = str((DATA_ROOT / "cuts_dev-clean-2.jsonl").resolve())

cmd = [
    "torchrun", "--nproc_per_node=2",
    "examples/speechlm2/salm_eval.py",
    "pretrained_name=salm_tutorial_hf/",
    f"inputs={DEV_CUTS}",
    "batch_size=8",
    "max_new_tokens=128",
    "ep_size=2",
    "output_manifest=tutorial_generations.jsonl",
    "user_prompt=Transcribe the following: <|audioplaceholder|>",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 10. Inspect Results

In [ ]:
import json

with open("tutorial_generations.jsonl") as f:
    results = [json.loads(line) for line in f]

for r in results[:5]:
    print(f"REF: {r['text']}")
    print(f"HYP: {r['pred_text']}")
    print()

print(f"Total examples: {len(results)}")

## Summary

In this tutorial we:

1. Downloaded Mini LibriSpeech with Lhotse
2. Trained SALMAutomodel with Nemotron Nano V3 MoE backbone (EP=2 on the FSDP data-parallel axis)
3. Converted the distributed checkpoint to HuggingFace format
4. Evaluated with distributed inference and computed WER

### Next Steps

- **Scale up**: more GPUs, larger datasets
- **Enable LoRA**: uncomment the `lora:` block for parameter-efficient LLM fine-tuning
- **Try other parallelism combos**: TP + EP, HSDP (`dp_replicate_size > 1`)
- See `docs/source/speechlm2/` for full documentation